# LabeledFewShot

Sample labeled training examples and attach them directly as demonstrations; no teacher calls are needed during compilation.

**Use it when:** You have trustworthy labels, want the cheapest few-shot baseline, and do not need generated reasoning traces.

**What compilation changes:** Demonstrations only; the original instruction remains unchanged.

Important configuration in this benchmark:

- `k=4` caps prompt growth
- `sample=True` samples from the frozen training split

Every notebook loads the canonical 300-row expanded dataset and frozen,
pair-grouped membership: 160 training, 60 validation, and 80 locked-test rows.
A semantic human/AI pair can never cross partitions. Optimizer choices use
validation only; the locked test is released once after the program is frozen.
These scores teach optimizer tradeoffs, not a general AI-detector leaderboard.

In [1]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "chapter06").is_dir() else cwd.parent
if not (REPO_ROOT / "chapter06" / "results" / "expanded_notebooks" / "comparison.json").exists():
    raise RuntimeError("Run this notebook from the repository or chapter06 directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from chapter06.notebook_support import artifact_paths, learned_program_preview, verify_prompt_artifact
from chapter06.optimizer_runtime import (
    format_result,
    load_frozen_examples,
    published_result,
    run_optimizer,
    split_summary,
)

OPTIMIZER = 'labeled-few-shot'
splits = load_frozen_examples()
RUN_LIVE = os.getenv("CHAPTER06_RUN_LIVE", "0") == "1"
print(f"optimizer={OPTIMIZER!r}; live={RUN_LIVE}")
print(split_summary(splits))

optimizer='labeled-few-shot'; live=False
train=160 (human=80, AI=80); validation=60 (human=30, AI=30); test=80 (human=40, AI=40)


## Compile shape

This is the essential DSPy call used by the shared executable runner:

```python
optimizer = dspy.LabeledFewShot(k=4)
optimized_detector = optimizer.compile(detector, trainset=trainset, sample=True)
```

`compile` returns a program. The shared runner then evaluates that program on the
untouched 80-row locked test split. The baseline has its own notebook; all other
notebooks report validation and locked-test accuracy separately.

In [2]:
if RUN_LIVE:
    live_run = run_optimizer(
        OPTIMIZER,
        splits=splits,
    )
    result = live_run.summary()
else:
    result = published_result(OPTIMIZER)

print(format_result(result))
print()
print(artifact_paths(OPTIMIZER))

optimizer: labeled-few-shot
task model: openai/gpt-5.6-luna
final test accuracy: 67.5% (54/80)
optimized validation accuracy: 63.3%
same-model baseline: 53.8%
uplift: +13.8 points
optimization cost: $0.0000
optimization time: 0.0s
mean inference latency: 1.621s
p95 inference latency: 2.482s

Published artifacts:
- program snapshot: chapter06/results/expanded_notebooks/labeled-few-shot/full/optimized_program.json
- prompt snapshot: chapter06/results/expanded_notebooks/labeled-few-shot/full/learned_prompt.json
- chapter comparison: chapter06/CHAPTER_RESULTS.md


## Read the result

Inspect the four saved demos. This method can improve in-context calibration, but sampled examples are not selected against the validation set.

The saved output above uses the checked-in expanded-dataset result, so opening or
rerunning the notebook is free. The paid run first passed a bounded smoke profile,
then froze its full program using training and validation only. Set
`CHAPTER06_RUN_LIVE=1` before launching Jupyter to reproduce that full protocol;
prompt optimizers require an OpenAI key, while weight optimizers also require the
local PyTorch/Transformers stack. The next cell previews the durable program artifact.

In [3]:
print(learned_program_preview(OPTIMIZER))
print()
print("Frozen program/prompt consistency:", verify_prompt_artifact(OPTIMIZER))

Predictor: detect.predict
Learned instruction (59 characters):
Decide whether the supplied passage was generated by an AI.

Demonstrations: 4
1. is_ai=False: 1. The browser makes a request: GET http://localhost:3000. 2. Our Rails application receives this request. 3. The Rails router maps the root route to the index action of Article…
2. is_ai=True: Datasets resemble RDDs, but, rather than relying on Java serialization or Kryo, they employ specialized Encoders to serialize objects for processing or transmission across the n…
3. is_ai=False: As well as grouping tasks into groups, you can also label the dependency edges between different tasks in the Graph view - this can be especially useful for branching areas of y…
4. is_ai=False: For example: in Kubernetes, a Deployment is an object that can represent an application running on your cluster. When you create the Deployment, you might set the Deployment spe…

Frozen program/prompt consistency: {'checked': True, 'predictors': 1, 'prompt_

## Apply the pattern

Adapt the compile shape above to your own DSPy program, metric, and frozen
train/validation split. Keep the test set untouched until the optimizer returns,
then report final test accuracy as `correct / test examples` so every optimizer is easy
to compare. Use the separate baseline notebook when you also need uplift.

The complete Chapter 6 rerun is summarized in `CHAPTER_RESULTS.md`; machine-readable
scores, prompts, programs, predictions, timing, cost, versions, hashes, failures,
and retries live under `results/expanded_notebooks/`. Weight-model payloads are
generated locally and Git-ignored; their checked-in manifests retain file hashes,
sizes, configuration, prompts, programs, and scores. Credentials and provider
request bodies are intentionally excluded.